# H51: đọc ma trận hữu thanh và kết quả F0

Notebook này đọc dữ liệu đã kiểm tra; không tự chạy lại 2.793 lượt học. Để tái lập toàn bộ thí nghiệm, dùng các lệnh trong `H51_RUNBOOK.md`. Các notebook gốc và file đã nộp giữ nguyên.

`Center` là cách gán nhãn chuẩn của khung bằng nhãn LAB tại điểm giữa cửa sổ. Nó không phải thuật toán nhận hữu thanh khi chạy trên file mới. Logistic/SVM/kNN/RF học nhãn V so với UV/SIL trong file train. GMM phân cụm không đọc LAB khi học hoặc gán nghĩa cụm; bước chọn pipeline qua validation vẫn dùng nhãn và ba thống kê chuẩn.

Năm nhóm: P=độ tuần hoàn; E=năng lượng; Z=ZCR; S=tỷ lệ phổ tần số cao; M=13 MFCC. Ma trận gồm 31 tổ hợp × 5 mô hình, seed 11/29/47. Điều kiện khôi phục và pitch dùng chung. Chỉ có bốn file train; seed không tạo thêm người nói.


In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd

cwd = Path.cwd()
candidates = [cwd, cwd / 'research_workbench_2026_10_07', cwd / 'XLTN-BT2' / 'research_workbench_2026_10_07']
workbench = next((p for p in candidates if (p / 'results/H51_external_verification.json').is_file()), None)
assert workbench is not None, 'Mở notebook từ folder workbench, repo XLTN-BT2 hoặc workspace XLTN local'
repo = workbench.parent
results = workbench / 'results'
for phase in ('train', 'external'):
    receipt = json.loads((results / f'H51_{phase}_verification.json').read_text(encoding='utf-8'))
    experiment_path = results / f'H51_{phase}_experiment.json'
    assert receipt['passed']
    assert hashlib.sha256(experiment_path.read_bytes()).hexdigest() == receipt['experiment_sha256']
    experiment = json.loads(experiment_path.read_text(encoding='utf-8'))
    for relative, digest in experiment['artifacts'].items():
        assert hashlib.sha256((repo / relative).read_bytes()).hexdigest() == digest, relative
freeze = json.loads((results / 'H51_FROZEN_SELECTION.json').read_text(encoding='utf-8'))
print('Verified train/external evidence. Final recipe:', freeze['recipe']['id'])
print('Grouped outer 4 folds / inner 3 folds; no random frame split; exploratory after repeated train use.')


Verified train/external evidence. Final recipe: gmm_PE
Grouped outer 4 folds / inner 3 folds; no random frame split; exploratory after repeated train use.


## Mục tiêu và kết quả kiểm định chéo

Mục tiêu là **mỗi file Average MAPE <2%**. MAPE là trung bình ba lỗi tương đối mean/std/count. Tập train đã được nghiên cứu nhiều vòng; test và KEELE cũng có lịch sử được xem. Không coi các kết quả này là đánh giá trên dữ liệu hoàn toàn mới.


In [2]:
metrics = pd.read_csv(results / 'H51_metrics.csv')
print(metrics[(metrics['split'] == 'nested') & (metrics['model'] == 'candidate')][['seed', 'file', 'option_id', 'average_mape', 'macro_f1', 'recall_v']].to_string(index=False))
print('\nFixed final recipe on 4 train + 4 descriptive test files:')
print(pd.read_csv(results / 'H51_all8_target.csv').to_string(index=False))


 seed          file option_id  average_mape  macro_f1  recall_v
   11  phone_F1.wav    gmm_PE      0.340080  0.938333  0.941176
   29  phone_F1.wav    gmm_PE      0.340080  0.938333  0.941176
   47  phone_F1.wav    gmm_PE      0.340080  0.938333  0.941176
   11  phone_M1.wav  gmm_PEZS      4.007407  0.933117  0.950820
   29  phone_M1.wav  gmm_PEZS      4.007407  0.933117  0.950820
   47  phone_M1.wav  gmm_PEZS      4.007407  0.933117  0.950820
   11 studio_F1.wav   gmm_PES      1.473576  0.900407  0.967480
   29 studio_F1.wav   gmm_PES      1.473576  0.900407  0.967480
   47 studio_F1.wav   gmm_PES      1.473576  0.900407  0.967480
   11 studio_M1.wav   gmm_PES      2.399893  0.817625  0.872340
   29 studio_M1.wav   gmm_PES      2.399893  0.817625  0.872340
   47 studio_M1.wav   gmm_PES      2.399893  0.817625  0.872340

Fixed final recipe on 4 train + 4 descriptive test files:
 seed  train_files_below2  test_files_below2  all8_below2
   11                   3                  0       

## Ma trận và seed

Ba seed thực sự đổi phép học của RF/GMM. Logistic/SVM/kNN cho kết quả xác định nên các seed chỉ lặp cùng kết quả. Độ lệch chuẩn qua seed không phải mức bất định về toàn bộ người nói.

![Ma trận MAPE và F1](figures/H51_matrix.png)


In [3]:
summary = pd.read_csv(results / 'H51_seed_summary.csv')
full = ['logistic_PEZSM', 'svm_PEZSM', 'knn_PEZSM', 'rf_PEZSM', 'gmm_PEZSM', 'hard170']
print(summary[summary['recipe_id'].isin(full)].to_string(index=False))
print('\nFull matrix rows:', len(summary), '; non-control recipes:', len(summary) - 1)


     recipe_id   method blocks  mean_mape  std_seed_mape  min_seed_mape  max_seed_mape  worst_file_any_seed  mean_f1  mean_recall_v
     gmm_PEZSM      gmm  PEZSM  10.440779        0.00000      10.440779      10.440779            15.468701 0.908091       0.955651
       hard170  control    NaN   1.124932        0.00000       1.124932       1.124932             1.909923 0.893977       0.929270
     knn_PEZSM      knn  PEZSM   6.001600        0.00000       6.001600       6.001600            10.961126 0.904031       0.958520
logistic_PEZSM logistic  PEZSM   8.736663        0.00000       8.736663       8.736663            17.097668 0.909073       0.963194
      rf_PEZSM       rf  PEZSM   8.797629        0.00997       8.786631       8.806077            16.565925 0.897855       0.955568
     svm_PEZSM      svm  PEZSM   9.018525        0.00000       9.018525       9.018525            16.943232 0.910579       0.966879

Full matrix rows: 156 ; non-control recipes: 155


## Ablation, tương tác và permutation

Leave-one-block-out bỏ một nhóm khỏi mô hình dùng toàn bộ đặc trưng. Tương tác dùng f(A+B)−f(A)−f(B)+f(control), là phép so sánh hiệu ứng cộng, không chứng minh nhân quả. Permutation đảo đầu vào của classifier, giữ nguyên điều kiện khôi phục và pitch.


In [4]:
removed = pd.read_csv(results / 'H51_leave_one_block_out.csv')
print(removed.groupby(['method', 'removed_block'])[['delta_mape', 'delta_f1']].mean().to_string())
permuted = pd.read_csv(results / 'H51_permutation.csv')
print('\nPermutation groups:', len(permuted))
print(permuted.groupby(['recipe_id', 'block'])[['delta_average_mape', 'delta_macro_f1']].mean().to_string())


                        delta_mape  delta_f1
method   removed_block                      
gmm      E                1.370497 -0.009504
         M               -7.970171 -0.011593
         P               -2.459365 -0.007863
         S               -3.532927 -0.011129
         Z               -4.528188 -0.004692
knn      E                1.533684  0.000120
         M                2.881595 -0.010587
         P               -0.614440 -0.007306
         S                0.000000  0.000000
         Z                0.977451 -0.001398
logistic E                4.146578 -0.019000
         M                0.206940 -0.004669
         P                0.023373  0.005056
         S                0.000000  0.000000
         Z                0.000000  0.000000
rf       E                5.801760  0.001738
         M                0.407406  0.004591
         P               -0.564716  0.007258
         S               -0.045959  0.003498
         Z                0.150446 -0.000245
svm      E

## Lỗi cần phân biệt

Một khung có nhãn V đúng chưa xác nhận F0 ước lượng của nó đúng. Khi thêm một cao độ xa mean cũ, std có thể tăng mạnh. F0num chuẩn và số khung tâm V cũng khác nhau; chưa biết protocol tạo ba thống kê của thầy.


In [5]:
print(pd.read_csv(results / 'H51_selected_recovery_cases.csv').to_string(index=False))
print('\nConditional count tradeoff, not a lower bound for all models:')
print(pd.read_csv(results / 'H51_label_count_tradeoff.csv').to_string(index=False))
print('\nSee H51_INTERPRETATION.md for variance decomposition and limits.')


         file recipe_id   time_s center_label  boundary  added_estimated_f0_hz  periodicity  relative_rms  zcr_per_second  high_frequency_ratio  nearest_baseline_time_s  nearest_baseline_estimated_f0_hz  distance_ms  per_frame_pitch_truth_available
 phone_M1.wav  gmm_PEZS 1.992500            v     False             240.172328     0.759910      0.296610     1122.807018              0.005188                 1.962500                         98.111447         30.0                            False
studio_M1.wav   gmm_PES 1.702494            v     False              91.632365     0.753645      0.585896      881.198910              0.074968                 1.692494                         91.455637         10.0                            False

Conditional count tradeoff, not a lower bound for all models:
         file  center_v_frames  reference_F0num  count_mape_if_exact_v_and_all_finite  average_mape_contribution_if_other_components_zero
 phone_F1.wav              153            148.0     

## Robustness và chuyển dữ liệu

Noise kế thừa nhãn và thống kê của file sạch là mục tiêu tiềm ẩn, không có F0 chuẩn mới sau thêm nhiễu. Phone↔studio chỉ có hai file mỗi điều kiện và bị trộn với khác biệt người nói. KEELE có reference từng khung nhưng cửa sổ và độ trễ khác pipeline; đây là kiểm tra chuyển dữ liệu để chẩn đoán, không đủ kết luận lỗi chỉ do BT2 ít dữ liệu.


In [6]:
noise = pd.read_csv(results / 'H51_robustness.csv')
print(noise.groupby(['condition', 'recipe_id'])[['average_mape', 'macro_f1', 'recall_v']].mean().to_string())
transfer = pd.read_csv(results / 'H51_keele_pooled.csv')
print('\nKEELE pooled per seed, keeping GPE/VDE/FFE/RPA together:')
print(transfer[['recipe_id', 'seed', 'gpe20_pct', 'vde_pct', 'ffe20_pct', 'rpa50_pct', 'file_mean_average_mape']].to_string(index=False))


                           average_mape  macro_f1  recall_v
condition  recipe_id                                       
pink_20dB  gmm_PE              1.488678  0.897633  0.931930
           gmm_PEZSM           4.144912  0.905648  0.944007
           hard170             1.488678  0.897633  0.931930
           knn_PEZSM           2.547320  0.903516  0.942567
           logistic_PEZSM      5.441534  0.906772  0.948300
           rf_PEZSM            5.247531  0.899022  0.944007
           svm_PEZSM           5.211047  0.902053  0.945641
white_20dB gmm_PE              1.364792  0.894086  0.927636
           gmm_PEZSM           3.222216  0.899712  0.933369
           hard170             1.364792  0.894086  0.927636
           knn_PEZSM           2.569276  0.898392  0.939298
           logistic_PEZSM      3.364022  0.902195  0.940322
           rf_PEZSM            3.303411  0.898157  0.940322
           svm_PEZSM           3.324160  0.900860  0.940322
white_30dB gmm_PE              1.231516 

## Đồ thị định trước cho bốn file

Logistic dùng toàn bộ đặc trưng, seed 11. Xanh lá=V, cam=UV, xanh nhạt=SIL theo LAB. Điểm F0 là ước lượng, không phải ground truth.

![phone_F1](figures/H51_qualitative_phone_F1.png)
![phone_M1](figures/H51_qualitative_phone_M1.png)
![studio_F1](figures/H51_qualitative_studio_F1.png)
![studio_M1](figures/H51_qualitative_studio_M1.png)

Đọc `H51_REPORT.md`, `H51_INTERPRETATION.md`, `EXPERIMENT_COVERAGE.md` để xem đầy đủ các cell, kết quả không đạt và hướng chưa thử. Không thay nhãn, tham số hoặc chọn seed từ notebook đọc kết quả này.
